# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [2]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [3]:
import os
import json
import shutil
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [4]:
load_dotenv()

True

In [5]:
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found in .env"
assert os.getenv("OPENAI_BASE_URL"), "OPENAI_BASE_URL not found in .env"
print("✓ Environment variables loaded successfully")

✓ Environment variables loaded successfully


In [5]:
shutil.rmtree("chromadb", ignore_errors=True)

### VectorDB Instance

In [6]:
chroma_client = chromadb.PersistentClient(path="chromadb")
print("✓ ChromaDB client initialized")

✓ ChromaDB client initialized


In [7]:
chroma_client.list_collections()

[Collection(name=udaplay_longterm), Collection(name=udaplay)]

In [7]:
api_key=os.getenv("OPENAI_API_KEY")
api_base=os.getenv("OPENAI_BASE_URL")

### Collection

In [8]:
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_base=api_base,
    api_key=api_key,
    model_name="text-embedding-3-small"
)
print("✓ Embedding function configured")

✓ Embedding function configured


In [9]:
collection = chroma_client.get_or_create_collection(
    name="udaplay",
    embedding_function=embedding_fn
)
print(f"✓ Collection 'udaplay' ready with {collection.count()} documents")

✓ Collection 'udaplay' ready with 0 documents


In [15]:
test_vec = embedding_fn(["test"])
print(len(test_vec[0]))

1536


In [1]:
chroma_client.list_collections()

NameError: name 'chroma_client' is not defined

In [10]:
collection.get()

{'ids': [],
 'embeddings': None,
 'documents': [],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': []}

### Add documents

In [11]:
# Make sure you have a directory "project/starter/games"
data_dir = "games"

# Check if collection is empty
if collection.count() == 0:
    print("Loading games into vector database...")
    
    for file_name in sorted(os.listdir(data_dir)):
        if not file_name.endswith(".json"):
            continue

        file_path = os.path.join(data_dir, file_name)
        with open(file_path, "r", encoding="utf-8") as f:
            game = json.load(f)

        # Create rich content for better semantic search
        content = f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']}"

        # Use file name (like 001) as ID
        doc_id = os.path.splitext(file_name)[0]

        collection.add(
            ids=[doc_id],
            documents=[content],
            metadatas=[game]
        )
        print(f"  ✓ Added: {game['Name']}")
    
    print(f"\n✓ Successfully loaded {collection.count()} games into the vector database!")
else:
    print(f"✓ Collection already contains {collection.count()} games")

Loading games into vector database...
  ✓ Added: Gran Turismo
  ✓ Added: Grand Theft Auto: San Andreas
  ✓ Added: Gran Turismo 5
  ✓ Added: Marvel's Spider-Man
  ✓ Added: Marvel's Spider-Man 2
  ✓ Added: Pokémon Gold and Silver
  ✓ Added: Pokémon Ruby and Sapphire
  ✓ Added: Super Mario World
  ✓ Added: Super Mario 64
  ✓ Added: Super Smash Bros. Melee
  ✓ Added: Wii Sports
  ✓ Added: Mario Kart 8 Deluxe
  ✓ Added: Kinect Adventures!
  ✓ Added: Minecraft
  ✓ Added: Halo Infinite

✓ Successfully loaded 15 games into the vector database!
